<a href="https://colab.research.google.com/github/LionPaul/retro-game-miner/blob/main/snes/notebooks/01_bronze_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
from io import StringIO
import os

# ==============================================================================
# CONFIGURAÇÃO DO PIPELINE (Mude as variáveis abaixo para mudar de console)
# ==============================================================================
CONSOLE_NAME = "snes"
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Super_Nintendo_Entertainment_System_games"

# Definição dos caminhos de salvamento seguindo a nossa arquitetura modular
DATASETS_DIR = "datasets"
OUTPUT_FILENAME = f"{CONSOLE_NAME}_games_raw.xlsx"
OUTPUT_PATH = os.path.join(DATASETS_DIR, OUTPUT_FILENAME)
# ==============================================================================

def pipeline_extracao_bronze(url, output_path):
    print(f"🎬 Iniciando a extração dos dados brutos para o console: {CONSOLE_NAME.upper()}")

    # Criar a pasta 'datasets' caso ela não exista no ambiente do Colab
    if not os.path.exists(DATASETS_DIR):
        os.makedirs(DATASETS_DIR)
        print(f"📁 Pasta '{DATASETS_DIR}' criada com sucesso.")

    # Definindo um User-Agent para a Wikipedia não bloquear a requisição
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        # 1. Faz o download do HTML da página
        print(f"🌐 Conectando à Wikipedia em: {url}")
        resposta = requests.get(url, headers=headers)
        resposta.raise_for_status()
        print("✅ Download do HTML concluído com sucesso.")

        # 2. Transforma o texto em um fluxo de dados e lê com o Pandas
        html_data = StringIO(resposta.text)
        tabelas = pd.read_html(html_data)

        # 3. Encontra a tabela principal (geralmente a que tem mais linhas)
        print("📊 Analisando tabelas da página para encontrar o catálogo de jogos...")
        tabela_jogos = max(tabelas, key=len)
        print(f"🎯 Tabela principal identificada com {len(tabela_jogos)} registros brutos.")

        # 4. Ajusta cabeçalhos em duas camadas (MultiIndex), comuns na Wikipedia
        if isinstance(tabela_jogos.columns, pd.MultiIndex):
            print("🔗 Combinando subcabeçalhos duplicados da tabela...")
            tabela_jogos.columns = [' '.join(col).strip() for col in tabela_jogos.columns.values]

        # 5. Salva o arquivo bruto sem aplicar nenhum filtro de exclusão
        print(f"💾 Salvando os dados brutos na camada Bronze...")
        tabela_jogos.to_excel(output_path, index=False)
        print(f"🎉 SUCESSO! Arquivo salvo em: {output_path}")

        # Retorna as primeiras linhas para validação visual no Colab
        return tabela_jogos.head(5)

    except Exception as e:
        print(f"❌ Erro crítico durante a extração: {e}")
        return None

# Executa a função
df_raw_preview = pipeline_extracao_bronze(WIKI_URL, OUTPUT_PATH)

🎬 Iniciando a extração dos dados brutos para o console: SNES
📁 Pasta 'datasets' criada com sucesso.
🌐 Conectando à Wikipedia em: https://en.wikipedia.org/wiki/List_of_Super_Nintendo_Entertainment_System_games
✅ Download do HTML concluído com sucesso.
📊 Analisando tabelas da página para encontrar o catálogo de jogos...
🎯 Tabela principal identificada com 1749 registros brutos.
🔗 Combinando subcabeçalhos duplicados da tabela...
💾 Salvando os dados brutos na camada Bronze...
🎉 SUCESSO! Arquivo salvo em: datasets/snes_games_raw.xlsx
